# Gold labeling — manual review of silver labels

In [2]:
import pandas as pd

In [ ]:
SILVER_CSV      = '../data\silver_labels\silver_labels_v1_2026_06_21.csv'
CLEAN_DATA_CSV = '../data/alliance/MDM_Population_cleaned_v5_2026_06_21.parquet'

In [26]:
# Read silver labels. get_silver_labels.ipynb wrote with the index, so drop any
# unnamed index column and force the PATID columns to string (ID dtype trap).
silver = pd.read_csv(SILVER_CSV)

In [67]:
records = pd.read_parquet(CLEAN_DATA_CSV)
records['BirthDT_clean'] = records['BirthDT_clean'].dt.strftime('%Y-%m-%d')

In [68]:
pairs = silver.merge(records.rename({c: c + '_A' for c in records.columns}, axis=1), 
             how='left', on='PATID_A')
pairs = pairs.merge(records.rename({c: c + '_B' for c in records.columns}, axis=1), 
             how='left', on='PATID_B')

In [69]:
pairs['gold_label'] = pd.NA

In [70]:
def show_pairs(df, k=5, fields=FIELDS):
    rows = []
    for _, r in df.head(k).iterrows():
        rows.append({'side':'A','PATID':r['PATID_A'], 'match_prob':round(r['match_prob'],4),
                     'scenario':r.get('scenario',''), **{f:r.get(f'{f}_A') for f in fields}})
        rows.append({'side':'B','PATID':r['PATID_B'], 'match_prob':'',
                     'scenario':'', **{f:r.get(f'{f}_B') for f in fields}})
        rows.append({'side':'', 'PATID':''})  # spacer
    return pd.DataFrame(rows).fillna('')

In [104]:
ALL_FIELDS = ['FirstNM_raw', 'LastNM_raw', 'MiddleNM_raw', 'SuffixNM_raw',
       'BirthDT_raw', 'SSN_raw', 'AddressLine1_raw', 'AddressLine2_raw',
       'CityNM_raw', 'ZipCD_raw', 'StateCD_raw', 'CountryNM',
       'PrimaryPhoneNBR_raw', 'Phone01NBR_raw', 'Phone02NBR_raw',
       'Phone03NBR_raw', 'Email_raw', 'SexAtBirthDSC_raw', 'FirstNM_clean',
       'LastNM_clean', 'MiddleNM_clean', 'SuffixNM_clean', 'BirthDT_clean',
       'SSN_clean', 'last_4_SSN', 'AddressLine1_clean', 'AddressLine2_clean',
       'CityNM_clean', 'ZipCD_clean_base', 'ZipCD_clean_ext', 'StateCD_clean',
       'PrimaryPhoneNBR_clean', 'Phone01NBR_clean', 'Phone02NBR_clean',
       'Phone03NBR_clean', 'Email_clean', 'SexAtBirthDSC_clean',
       'full_name_tokens', 'full_name_compact', 'Phones_set', 'Address_normalized']

CLEAN_FIELDS = ['FirstNM_clean',
       'LastNM_clean', 'MiddleNM_clean', 'SuffixNM_clean', 'BirthDT_clean',
       'SSN_clean', 'last_4_SSN', 'AddressLine1_clean', 'AddressLine2_clean',
       'CityNM_clean', 'ZipCD_clean_base', 'ZipCD_clean_ext', 'StateCD_clean',
       'Email_clean', 'SexAtBirthDSC_clean', 'Phones_set']

def show_pairs(df, k=50, fields=CLEAN_FIELDS):
    rows = []
    pairs = df.head(k).reset_index(drop=True)
    for _, r in pairs.iterrows():
        rows.append({'silver_label': r['silver_label'], **{f:r.get(f'{f}_A') for f in fields}})
        rows.append({'silver_label': r['silver_label'], **{f:r.get(f'{f}_B') for f in fields}})
        rows.append({'silver_label':''})  # spacer

    disp = pd.DataFrame(rows)

    def is_missing(v):
        return v is None or (isinstance(v, float) and pd.isna(v)) or (isinstance(v, str) and v.strip() == '')

    # color matrix matching disp shape
    colors = pd.DataFrame('', index=disp.index, columns=disp.columns)
    for i, r in pairs.iterrows():
        a_idx, b_idx = i * 3, i * 3 + 1  # display rows for this pair
        c = '#00C853' if r['silver_label'] else '#D50000'
        style = f'background-color: {c}'
        colors.loc[a_idx, 'silver_label'] = style
        colors.loc[b_idx, 'silver_label'] = style
        for f in fields:
            a, b = r.get(f'{f}_A'), r.get(f'{f}_B')
            am, bm = is_missing(a), is_missing(b)
            if am and bm:
                c = '#d3d3d3'  # grey  - both missing
            elif am or bm:
                c = '#fff3a3'  # yellow - one missing
            elif isinstance(a, list):
                if len(set(a + b)) == len(a) + len(b):
                    c = '#f4a8a8'  # red    - different
                else:
                    c = '#a8e6a3'  # green  - equal
            elif str(a) == str(b):
                c = '#a8e6a3'  # green  - equal
            else:
                c = '#f4a8a8'  # red    - different
            style = f'background-color: {c}'
            colors.loc[a_idx, f] = style
            colors.loc[b_idx, f] = style
    
    disp = disp.fillna('')

    # strip the _clean suffix on both frames together
    rename_map = {c: c.replace('_clean', '') for c in disp.columns}
    disp = disp.rename(columns=rename_map)
    colors = colors.rename(columns=rename_map)

    return disp.style.apply(lambda _: colors, axis=None)

In [ ]:
show_pairs(pairs, k=5)

In [ ]:
subset = pairs[(pairs['SSN_clean_A']==pairs['SSN_clean_B'])&(pairs['silver_label']==False)]
show_pairs(subset)